# 02. 청킹 → 임베딩 → 벡터 검색

01에서 만든 `parsed.jsonl` 을 검색 가능한 벡터 DB(Chroma)로 만든다.

효율 포인트 세 가지를 코드로 확인한다:
- **배치 인코딩** — 청크를 묶어서 한 번에 임베딩
- **콘텐츠 해시 id + upsert** — 재실행해도 중복이 안 쌓이는 멱등 인제스트
- **정규화 + 코사인** — normalize 후 저장

> 첫 실행 시 임베딩 모델(multilingual-e5-small, 약 470MB)을 다운로드한다.

In [3]:
import json
from pathlib import Path

records = [json.loads(line) for line in open("../data/parsed.jsonl", encoding="utf-8")]
print(len(records), "records")

6 records


## 1. 청킹

- PDF 본문 → RecursiveCharacterTextSplitter 로 분할
- Excel 행 → 이미 행 단위라 그대로 청크 1개
- 모든 청크에 **문서 제목을 prepend** — 짧은 청크의 맥락 손실 방지 (contextual chunking)

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)

chunks = []  # {text, metadata}
for rec in records:
    title = rec["metadata"]["title"]
    if rec["type"] == "pdf":
        for piece in splitter.split_text(rec["text"]):
            chunks.append({"text": f"[{title}] {piece}", "metadata": {**rec["metadata"], "source": rec["source"]}})
    else:  # excel-row: 행 자체가 이미 검색 단위
        chunks.append({"text": f"[{title}] {rec['text']}", "metadata": {**rec["metadata"], "source": rec["source"]}})

print(len(chunks), "chunks")
print(chunks[0]["text"][:150])

7 chunks
[CX-200 매뉴얼] 1. 개요
본 매뉴얼은 컨베이어 제어 시스템 CX-200의 설치와 운영 절차를 설명한다. CX-200은 PLC 기반 제어기로,
Mitsubishi 3E Frame 프로토콜로 통신한다.
2. 설치
장비는 반드시 전원 차단 상태에서 설치한다.  제어


## 2. 임베딩 (배치 + 정규화)

e5 계열은 **prefix 규약**이 있다 — 문서는 `passage: `, 질의는 `query: ` 를 붙여야 제 성능이 나온다.

`batch_size` 로 묶어 인코딩하고 `normalize_embeddings=True` 로 단위 벡터화한다 (코사인 = 내적).

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")

texts = [c["text"] for c in chunks]
embeddings = model.encode(
    [f"passage: {t}" for t in texts],
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings.shape  # (청크 수, 384)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(7, 384)

## 3. 저장 — 해시 id + upsert (멱등 인제스트)

청크 텍스트의 sha256 을 id 로 쓰면:
- 같은 내용을 다시 넣어도 **덮어쓰기**라 중복이 안 쌓임
- 문서가 수정되면 바뀐 청크만 새 id 로 추가됨 (증분 인제스트의 기초)

이 셀을 **두 번 실행해도 count 가 안 늘어나는 것**을 직접 확인해볼 것.

In [6]:
import hashlib
import chromadb

client = chromadb.PersistentClient(path="../chroma-db")
collection = client.get_or_create_collection("docs", metadata={"hnsw:space": "cosine"})

ids = [hashlib.sha256(t.encode()).hexdigest()[:32] for t in texts]

collection.upsert(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[c["metadata"] for c in chunks],
)
print("collection count:", collection.count())

collection count: 7


## 4. 검색

질의에는 `query: ` prefix. PDF 내용과 Excel 행이 **같은 컬렉션에서 함께** 검색되는 것을 확인한다.

In [7]:
def search(question: str, k: int = 3):
    q_emb = model.encode(f"query: {question}", normalize_embeddings=True)
    res = collection.query(query_embeddings=[q_emb.tolist()], n_results=k)
    print(f"Q: {question}")
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"  [{dist:.3f}] ({meta['source']}) {doc[:80]}")
    print()

search("알람 E-04 가 뜨면 어떻게 해야 해?")
search("모터 베어링 재고 몇 개 남았지?")
search("통신 케이블 설치할 때 주의사항")

Q: 알람 E-04 가 뜨면 어떻게 해야 해?
  [0.141] (cx200_manual.pdf) [CX-200 매뉴얼] 시스템의 지시에 따라 동작한다. 이상 발생 시 알람 코드가 HMI에 표시된다.
4. 유지보수
벨트 장력은 월 1회 점검한
  [0.198] (spare_parts.xlsx) [예비부품 목록] 부품명: 비상정지 스위치 | 부품코드: SW-ESTOP | 재고수량: 5 | 단가(원): 32000 | 교체주기: 점검시
  [0.206] (spare_parts.xlsx) [예비부품 목록] 부품명: 근접센서 PR-08 | 부품코드: SNS-PR08 | 재고수량: 20 | 단가(원): 27000 | 교체주기: 고장시

Q: 모터 베어링 재고 몇 개 남았지?
  [0.120] (spare_parts.xlsx) [예비부품 목록] 부품명: 모터 베어링 6204 | 부품코드: BRG-6204 | 재고수량: 8 | 단가(원): 15000 | 교체주기: 6개월
  [0.176] (cx200_manual.pdf) [CX-200 매뉴얼] 시스템의 지시에 따라 동작한다. 이상 발생 시 알람 코드가 HMI에 표시된다.
4. 유지보수
벨트 장력은 월 1회 점검한
  [0.183] (spare_parts.xlsx) [예비부품 목록] 부품명: 컨베이어 벨트 B-1200 | 부품코드: BLT-1200 | 재고수량: 3 | 단가(원): 450000 | 교체주기:

Q: 통신 케이블 설치할 때 주의사항
  [0.155] (cx200_manual.pdf) [CX-200 매뉴얼] 1. 개요
본 매뉴얼은 컨베이어 제어 시스템 CX-200의 설치와 운영 절차를 설명한다. CX-200은 PLC 기반 제어
  [0.175] (cx200_manual.pdf) [CX-200 매뉴얼] 시스템의 지시에 따라 동작한다. 이상 발생 시 알람 코드가 HMI에 표시된다.
4. 유지보수
벨트 장력은 월 1회 점검한
  [0.187] (spare_parts.xlsx) [예비부품 목록] 부품명: 비상정지 스위치

## 5. 메타데이터 필터

조건이 명확하면 벡터 검색 전에 메타데이터로 거른다 — 정확도와 속도 모두 개선.

In [8]:
q_emb = model.encode("query: 교체 주기가 짧은 부품", normalize_embeddings=True)
res = collection.query(
    query_embeddings=[q_emb.tolist()],
    n_results=3,
    where={"sheet": "예비부품"},  # Excel 행만 대상으로
)
res["documents"][0]

['[예비부품 목록] 부품명: 컨베이어 벨트 B-1200 | 부품코드: BLT-1200 | 재고수량: 3 | 단가(원): 450000 | 교체주기: 24개월',
 '[예비부품 목록] 부품명: 모터 베어링 6204 | 부품코드: BRG-6204 | 재고수량: 8 | 단가(원): 15000 | 교체주기: 6개월',
 '[예비부품 목록] 부품명: 근접센서 PR-08 | 부품코드: SNS-PR08 | 재고수량: 20 | 단가(원): 27000 | 교체주기: 고장시']

## 6. (선택) Langfuse 로 검색 트레이싱

observability 실습과 연결 — Langfuse 스택이 떠 있으면 검색 파이프라인을 trace 로 기록해서
질의별 검색 결과·지연을 UI 에서 확인할 수 있다. (`retriever` observation 타입 사용)

In [9]:
from dotenv import load_dotenv

# llmops-notes 레포의 Langfuse 실습 .env 를 사용 (경로는 환경에 맞게 수정)
ENV_PATH = Path(r"{CHANGEME}\langfuse\.env")
assert ENV_PATH.exists(), f".env 경로를 먼저 수정하세요: {ENV_PATH}"
load_dotenv(ENV_PATH)

from langfuse import get_client

lf = get_client()

question = "인버터 과열 조치 방법"
with lf.start_as_current_observation(name="doc-search", as_type="retriever", input={"q": question}) as obs:
    q_emb = model.encode(f"query: {question}", normalize_embeddings=True)
    res = collection.query(query_embeddings=[q_emb.tolist()], n_results=3)
    obs.update(output={"docs": res["documents"][0]})

lf.flush()
print("trace sent — Langfuse UI 에서 확인")

trace sent — Langfuse UI 에서 확인


In [10]:
from dotenv import load_dotenv

ENV_PATH = Path(r"{CHANGEME}\langfuse\.env")
assert ENV_PATH.exists(), f".env 경로를 먼저 수정하세요: {ENV_PATH}"
load_dotenv(ENV_PATH)

from langfuse import observe, get_client
from langfuse.langchain import CallbackHandler
from langchain_ollama import ChatOllama

lf = get_client()
llm = ChatOllama(model="gemma4:e4b", base_url="http://localhost:11434")
handler = CallbackHandler()


@observe(as_type="retriever")
def retrieve(question: str, k: int = 3) -> list[str]:
    q_emb = model.encode(f"query: {question}", normalize_embeddings=True)
    res = collection.query(query_embeddings=[q_emb.tolist()], n_results=k)
    return res["documents"][0]


@observe()
def rag_answer(question: str) -> str:
    docs = retrieve(question)  # 자식 retriever observation 으로 기록됨
    prompt = (
        "다음 문서를 참고해서 질문에 답하라. 문서에 없는 내용은 지어내지 말 것.\n\n"
        "== 문서 ==\n"
        + "\n".join(f"- {d}" for d in docs)
        + f"\n\n== 질문 ==\n{question}"
    )
    resp = llm.invoke(prompt, config={"callbacks": [handler]})  # generation 으로 기록됨
    return resp.content


print(rag_answer("인버터 과열이 발생하면 어떻게 조치해야 해?"))
lf.flush()

알람 코드 E-04는 인버터 과열이므로, 다음과 같이 조치해야 합니다.

1. **냉각팬 필터를 청소합니다.**
2. 냉각팬 필터 청소 후에도 증상이 재발할 경우 **인버터를 교체**해야 합니다.
